# Qwen3.5 msModelSlim Quantization Verification

This notebook follows the official Ascend `msmodelslim` Qwen3.5 example and verifies that the image has the required pieces in place for model quantization.
The image is validated against the working stack used for a successful `Qwen3.5-27B` run: `msmodelslim 26.0.0a2`, `transformers 5.2.0`, `torchvision 0.24.0`, `mistral-common 1.11.0`, `easydict 1.13`, and `wcmatch 10.1`.

Official reference:
- https://raw.gitcode.com/Ascend/msmodelslim/raw/master/example/Qwen3_5/README.md

This notebook does not start a large quantization job by default. It checks the runtime, prepares the official `msmodelslim quant` command, and only runs it when `RUN_QUANT = True`.

## Official Notes

The official Qwen3.5 guide states:
- `msmodelslim` must be installed.
- `transformers==5.2.0` is required.
- For the verified `Qwen3.5-27B` multimodal path in this image, `torchvision==0.24.0`, `mistral-common==1.11.0`, `easydict==1.13`, and `wcmatch==10.1` are also required at runtime.
- Supported devices are Atlas A2 and Atlas A3 training/inference products.
- Example command format:

```bash
msmodelslim quant --model_path ${MODEL_PATH} --save_path ${SAVE_PATH} --device npu --model_type Qwen3.5-27B --quant_type w8a8 --trust_remote_code True
```

In [ ]:
from pathlib import Path

# Update these paths before running a real quantization job.
MODEL_PATH = Path('/opt/app-root/src/models/Qwen3.5-27B')
SAVE_PATH = Path('/opt/app-root/src/output/qwen35-27b-w8a8')

# Officially documented model types in the Qwen3.5 README.
MODEL_TYPE = 'Qwen3.5-27B'
QUANT_TYPE = 'w8a8'
DEVICE = 'npu'
TRUST_REMOTE_CODE = True

# Safety switch: keep this False for environment verification.
RUN_QUANT = False

CANN_ENV = '/usr/local/Ascend/cann/set_env.sh'
ATB_ENV = '/usr/local/Ascend/nnal/atb/set_env.sh'

SUPPORTED_MODEL_TYPES = {
    'Qwen3.5-397B-A17B': {'w8a8', 'w4a8'},
    'Qwen3.5-122B-A10B': {'w8a8'},
    'Qwen3.5-35B-A3B': {'w8a8'},
    'Qwen3.5-27B': {'w8a8'},
}

print('MODEL_PATH =', MODEL_PATH)
print('SAVE_PATH  =', SAVE_PATH)
print('MODEL_TYPE =', MODEL_TYPE)
print('QUANT_TYPE =', QUANT_TYPE)
print('RUN_QUANT  =', RUN_QUANT)

In [ ]:
import os
import shlex
import stat
import subprocess


def run_cmd(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    env_prefix = f'source {CANN_ENV} && source {ATB_ENV}'
    full_cmd = f'{env_prefix} && {cmd}'
    print(f'$ {cmd}')
    result = subprocess.run(
        ['bash', '-lc', full_cmd],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}')
    return result


def shell_quote(value: str) -> str:
    return shlex.quote(value)


def find_writable_paths(root: Path, recursive: bool, limit: int = 20) -> list[str]:
    if not root.exists():
        return []

    offenders = []

    def inspect(candidate: Path) -> bool:
        try:
            mode = candidate.stat().st_mode
        except FileNotFoundError:
            return False
        if mode & (stat.S_IWGRP | stat.S_IWOTH):
            offenders.append(f'{candidate} mode={oct(mode & 0o777)}')
            return len(offenders) >= limit
        return False

    if inspect(root):
        return offenders
    if recursive and root.is_dir():
        for candidate in root.rglob('*'):
            if inspect(candidate):
                break
    return offenders


def prepare_msmodelslim_permissions(model_path: Path, save_path: Path) -> None:
    run_cmd(f'mkdir -p {shell_quote(str(save_path))}')
    for path in (model_path.parent, save_path.parent):
        if path.exists():
            run_cmd(f'chmod go-w {shell_quote(str(path))}', check=False)
    for path in (model_path, save_path):
        if path.exists():
            run_cmd(f'chmod -R go-w {shell_quote(str(path))}', check=False)


def print_msmodelslim_permission_report(model_path: Path, save_path: Path) -> bool:
    checks = [
        ('model parent', find_writable_paths(model_path.parent, recursive=False)),
        ('model tree', find_writable_paths(model_path, recursive=True)),
        ('save parent', find_writable_paths(save_path.parent, recursive=False)),
        ('save tree', find_writable_paths(save_path, recursive=True)),
    ]

    print('Permission preflight:')
    for label, offenders in checks:
        if offenders:
            print(f'  [{label}] writable path(s) still present, showing up to {len(offenders)}:')
            for offender in offenders:
                print(f'    {offender}')
        else:
            print(f'  [{label}] OK')

    return not any(offenders for label, offenders in checks if label.startswith('model'))


print('Helper functions loaded')

In [ ]:
print('Python version:')
run_cmd('python3 --version')

print('Checking verified runtime stack:')
run_cmd("python3 - <<'PY'\nimport importlib.metadata as m\nfor name in ['bracex', 'easydict', 'msmodelslim', 'transformers', 'huggingface-hub', 'torchvision', 'mistral-common', 'wcmatch']:\n    print(name, m.version(name))\nfrom huggingface_hub import is_offline_mode\nprint('huggingface-hub is_offline_mode ok', is_offline_mode())\nimport bracex\nimport easydict\nimport mistral_common\nimport msmodelslim\nimport torchvision\nimport transformers\nimport wcmatch\nprint('runtime imports ok')\nPY")

print('Checking transformers version:')
run_cmd("python3 -c \"import transformers; print(transformers.__version__)\"")

print('Checking msmodelslim CLI:')
run_cmd('msmodelslim --help | head -n 20')

In [ ]:
if MODEL_TYPE not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f'Unsupported MODEL_TYPE: {MODEL_TYPE}')

if QUANT_TYPE not in SUPPORTED_MODEL_TYPES[MODEL_TYPE]:
    raise ValueError(f'{MODEL_TYPE} does not support quant type {QUANT_TYPE} in the official README')

SAVE_PATH.mkdir(parents=True, exist_ok=True)

quant_cmd = ' '.join([
    'msmodelslim', 'quant',
    '--model_path', shell_quote(str(MODEL_PATH)),
    '--save_path', shell_quote(str(SAVE_PATH)),
    '--device', DEVICE,
    '--model_type', MODEL_TYPE,
    '--quant_type', QUANT_TYPE,
    '--trust_remote_code', str(TRUST_REMOTE_CODE),
])

print('Official quant command:')
print(quant_cmd)
print('Permission prep that will run when RUN_QUANT = True:')
print(f'  chmod go-w {MODEL_PATH.parent}')
print(f'  chmod -R go-w {MODEL_PATH}')
print(f'  mkdir -p {SAVE_PATH}')
print(f'  chmod go-w {SAVE_PATH.parent}')
print(f'  chmod -R go-w {SAVE_PATH}')

if MODEL_PATH.exists():
    print(f'Model path exists: {MODEL_PATH}')
    if not print_msmodelslim_permission_report(MODEL_PATH, SAVE_PATH):
        print('MODEL_PATH is not ready yet. RUN_QUANT = True will try to fix permissions and re-check before quantization.')
else:
    print(f'Model path does not exist yet: {MODEL_PATH}')
    print('Update MODEL_PATH before setting RUN_QUANT = True.')

## Supported Official Qwen3.5 Variants

- `Qwen3.5-397B-A17B`: `w8a8`, `w4a8`
- `Qwen3.5-122B-A10B`: `w8a8`
- `Qwen3.5-35B-A3B`: `w8a8`
- `Qwen3.5-27B`: `w8a8`

If you want to verify a different official model, change `MODEL_TYPE`, `MODEL_PATH`, `SAVE_PATH`, and `QUANT_TYPE` above.

In [ ]:
if RUN_QUANT:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'MODEL_PATH does not exist: {MODEL_PATH}')
    prepare_msmodelslim_permissions(MODEL_PATH, SAVE_PATH)
    if not print_msmodelslim_permission_report(MODEL_PATH, SAVE_PATH):
        raise RuntimeError(
            'MODEL_PATH still has group/other writable bits after permission prep. '
            'This usually means the mounted model directory was created with permissive modes '
            'or the storage backend is not honoring chmod.'
        )
    run_cmd(quant_cmd)
else:
    print('RUN_QUANT is False; skipping the real quantization job.')
    print('Set RUN_QUANT = True after you prepare the model weights on disk.')